# Module 01 — Autograd Engine

This notebook is about exactly one thing: **automatic differentiation**. Not neural networks yet, not training yet — just the mechanism that makes both possible.

**The core idea:** a neural network is a mathematical expression built out of simple operations (add, multiply, etc.) applied to numbers. Training means adjusting the network's parameters to make the expression's output (the loss) smaller. To know *which direction* to adjust each parameter, we need its **gradient** — how much the output changes per tiny change in that parameter. Backpropagation computes every parameter's gradient in one backward sweep over the expression, using the chain rule from calculus, one operation at a time.

We'll:
1. Build a `Value` class that wraps a number and remembers how it was computed (a computation graph).
2. Give it backward rules for `+`, `*`, `**`, `tanh`, `relu` — enough primitives to build any neural net later.
3. Verify it against a hand-computed derivative, so we trust it.

That's it for this notebook. Module 02 uses this engine to build actual network structure (neurons, layers); this one only builds and tests the engine itself.

In [ ]:
import math

## The `Value` class

Every `Value` wraps one number and stores:
- `data` — the actual number.
- `grad` — how much the *final output of the whole graph* changes per unit change in this value. Starts at 0, gets filled in during the backward pass.
- `_prev` — the other `Value`s combined to produce this one (its parents in the computation graph).
- `_backward` — a tiny function that knows how to push gradient from this `Value` back onto its parents, for whichever operation created it.

The backward rule for each operation is just its derivative, applied via the chain rule:
- `out = a + b` → `d(out)/da = 1`, `d(out)/db = 1`. Gradient passes straight through to both parents, unchanged.
- `out = a * b` → `d(out)/da = b`, `d(out)/db = a`. Each parent's gradient is the *other* parent's value, scaled by however much gradient is flowing into `out`.
- `out = a ** p` → `d(out)/da = p * a ** (p - 1)`, the ordinary power rule.
- `out = tanh(a)` → `d(out)/da = 1 - tanh(a)**2`.
- `out = relu(a)` → `d(out)/da = 1` if `a > 0` else `0`.

In every `_backward` closure below, notice the pattern `parent.grad += local_derivative * out.grad`: that's the chain rule — local derivative times whatever gradient already arrived from downstream. We use `+=` (not `=`) because a `Value` can feed into more than one downstream computation, and its total gradient is the *sum* of the gradient contributions from every path it feeds into (multivariable chain rule).

In [ ]:
class Value:
    """A scalar that remembers how it was computed, so gradients can flow backward through it."""

    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, power):
        assert isinstance(power, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** power, (self,), f"**{power}")

        def _backward():
            self.grad += (power * self.data ** (power - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    def backward(self):
        # Topological sort: order every node so each one appears only after
        # all the nodes that depend on it. That guarantees a node's gradient
        # is fully accumulated (from every downstream path) before we use it
        # to push gradient further back to its own parents.
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        self.grad = 1.0  # d(self)/d(self) = 1, the seed for the chain rule
        for v in reversed(topo):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

## Sanity check

Before trusting this for anything, verify it against a derivative we can compute by hand.

$y = x^2 + 2x + 1 \implies \dfrac{dy}{dx} = 2x + 2$. At $x = 3$: $\dfrac{dy}{dx} = 8$.

In [ ]:
x = Value(3.0)
y = x * x + 2 * x + 1
y.backward()

print(f"y.data  = {y.data}   (expected 16.0)")
print(f"x.grad  = {x.grad}   (expected 8.0)")
assert y.data == 16.0
assert abs(x.grad - 8.0) < 1e-9
print("matches the hand-computed derivative")

## A second check, with branching

The `+=` in every `_backward` matters most when a value is *reused* — fed into more than one downstream computation. Check that case specifically, since it's the one naive implementations get wrong.

$z = x \cdot x + x \implies \dfrac{dz}{dx} = 2x + 1$. At $x = 4$: $\dfrac{dz}{dx} = 9$. Here `x` is used twice — once in the multiplication, once in the addition — so its gradient must be the *sum* of both contributions.

In [ ]:
x = Value(4.0)
z = x * x + x
z.backward()

print(f"z.data  = {z.data}   (expected 20.0)")
print(f"x.grad  = {x.grad}   (expected 9.0)")
assert z.data == 20.0
assert abs(x.grad - 9.0) < 1e-9
print("gradient correctly accumulated across both uses of x")

## What just happened

We built a **computation graph** (`Value` objects linked via `_prev`) and a **backward pass** that applies the chain rule automatically, node by node, in reverse topological order. That's the entire mechanism autograd libraries (including PyTorch's) are built on — they just do it with tensors instead of scalars, in compiled code, on a GPU.

**Next: Module 02** — use this engine to build actual network structure: a `Neuron`, a `Layer` of neurons, and an `MLP` (stack of layers). No training yet — just forward passes, to focus purely on how a neural network is put together.